In [2]:
# === 새 노트북: 거래단위 2단계 stacking 검증 ===
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

FEAT_DIR = '/content/drive/MyDrive/26-1 머신러닝 팀플/feature'
DATA_DIR = '/content/drive/MyDrive/26-1 머신러닝 팀플/competition_data'   # train_transactions.csv, y_train.csv 있는 폴더 (glob으로 확인한 그 경로)
N_SPLITS = 5

# --- 로드 ---
def load(p):
    d = pd.read_csv(f'{DATA_DIR}/{p}', encoding='cp949')
    dt = pd.to_datetime(d['sales_datetime'])
    d['hour'] = dt.dt.hour
    d['weekday'] = dt.dt.weekday
    d['month'] = dt.dt.month
    d['is_weekend'] = (dt.dt.weekday>=5).astype(int)
    return d
tr_tx = load('train_transactions.csv')
te_tx = load('test_transactions.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')
fold = pd.read_parquet(f'{FEAT_DIR}/fold.parquet').set_index('custid')['fold']
allf = pd.read_parquet(f'{FEAT_DIR}/feats_1st_plus_8adv.parquet')   # 439 (고객단위)

# --- 거래 레벨 피쳐 (범주형은 빈도 기반 인코딩 + 금액/시간) ---
CAT = ['brd_nm','corner_nm','pc_nm','part_nm','str_nm','buyer_nm']
NUM = ['tot_amt','dis_amt','net_amt','inst_mon','hour','weekday','month','is_weekend']

def freq_encode(tr, te, col):
    # 거래 빈도 인코딩 (train+test 합쳐 계산, 비지도라 누수 아님)
    vc = pd.concat([tr[col], te[col]]).astype(str).value_counts()
    return tr[col].astype(str).map(vc).fillna(0).values, te[col].astype(str).map(vc).fillna(0).values

def tx_level_features(tr, te):
    Xtr = pd.DataFrame(index=tr.index); Xte = pd.DataFrame(index=te.index)
    for c in NUM:
        Xtr[c] = tr[c].fillna(0).values; Xte[c] = te[c].fillna(0).values
    for c in CAT:
        a,b = freq_encode(tr, te, c); Xtr[f'freq_{c}']=a; Xte[f'freq_{c}']=b
    Xtr['disc_rate'] = tr['dis_amt']/(tr['tot_amt'].abs()+1)
    Xte['disc_rate'] = te['dis_amt']/(te['tot_amt'].abs()+1)
    return Xtr, Xte

Xtx_tr, Xtx_te = tx_level_features(tr_tx, te_tx)

# 거래에 고객 성별 + 고객 fold 부착
ymap = y_train.set_index('custid')['gender']
tr_tx['g'] = tr_tx['custid'].map(ymap)
tr_tx['fold'] = tr_tx['custid'].map(fold)

# --- 거래 레벨 OOF (고객 fold 기준, 누수 차단) ---
tx_oof = np.zeros(len(tr_tx))          # train 거래별 남성확률 (OOF)
tx_test_pred = np.zeros(len(te_tx))    # test 거래별 남성확률
fvals = tr_tx['fold'].values
for i in range(N_SPLITS):
    trm = fvals != i
    vam = fvals == i
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0, n_jobs=-1, max_bin=255, verbose=-1)
    m.fit(Xtx_tr.values[trm], tr_tx['g'].values[trm])
    tx_oof[vam] = m.predict_proba(Xtx_tr.values[vam])[:,1]
    tx_test_pred += m.predict_proba(Xtx_te.values)[:,1] / N_SPLITS

print('거래레벨 OOF AUC (거래단위):', round(roc_auc_score(tr_tx['g'], tx_oof), 5))

# --- 거래별 확률 → 고객 단위 분포 집계 ---
def agg_tx_prob(tx_df, probs):
    d = pd.DataFrame({'custid': tx_df['custid'].values, 'p': probs})
    g = d.groupby('custid')['p']
    out = g.agg(['mean','std','max','min','median',
                 lambda x:x.quantile(.25), lambda x:x.quantile(.75)])
    out.columns = ['txp_mean','txp_std','txp_max','txp_min','txp_median','txp_q25','txp_q75']
    out['txp_spread'] = out['txp_q75'] - out['txp_q25']
    out['txp_frac_male'] = g.apply(lambda x:(x>0.5).mean())   # 남성성 거래 비율
    return out.fillna(0)

stk_tr = agg_tx_prob(tr_tx, tx_oof)
stk_te = agg_tx_prob(te_tx, tx_test_pred)
stack_feats = pd.concat([stk_tr, stk_te])
print('stacking 피쳐:', stack_feats.shape[1], '컬럼:', list(stack_feats.columns))

# --- 439 + stacking OOF 비교 ---
tr_ids = y_train['custid'].values
def oof_of(feats):
    X = feats.loc[tr_ids]; t = ymap.loc[tr_ids].values
    fv = pd.Series(tr_ids, index=tr_ids).map(fold).values
    Xv = X.values.astype(np.float32); o = np.zeros(len(Xv))
    for i in range(N_SPLITS):
        m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=31, subsample=0.8,
            colsample_bytree=0.6, reg_lambda=2.0, n_jobs=-1, max_bin=127, verbose=-1)
        m.fit(Xv[fv!=i], t[fv!=i]); o[fv==i] = m.predict_proba(Xv[fv==i])[:,1]
    return roc_auc_score(t, o)

print('BASE 439          :', round(oof_of(allf), 5))
print('439 + tx-stacking :', round(oof_of(allf.join(stack_feats)), 5))

# 저장 (효과 있으면 AutoGluon에 쓰게)
stack_feats.to_parquet(f'{FEAT_DIR}/tx_stack_feats.parquet')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
거래레벨 OOF AUC (거래단위): 0.62747
stacking 피쳐: 9 컬럼: ['txp_mean', 'txp_std', 'txp_max', 'txp_min', 'txp_median', 'txp_q25', 'txp_q75', 'txp_spread', 'txp_frac_male']
BASE 439          : 0.71373
439 + tx-stacking : 0.71459


In [4]:
from sklearn.model_selection import StratifiedKFold

def load(p):
    d = pd.read_csv(f'{DATA_DIR}/{p}', encoding='cp949')
    dt = pd.to_datetime(d['sales_datetime'])
    d['hour']=dt.dt.hour; d['weekday']=dt.dt.weekday; d['month']=dt.dt.month
    d['is_weekend']=(dt.dt.weekday>=5).astype(int)
    return d
tr_tx=load('train_transactions.csv'); te_tx=load('test_transactions.csv')
y_train=pd.read_csv(f'{DATA_DIR}/y_train.csv')
fold=pd.read_parquet(f'{FEAT_DIR}/fold.parquet').set_index('custid')['fold']
allf=pd.read_parquet(f'{FEAT_DIR}/feats_1st_plus_8adv.parquet')
ymap=y_train.set_index('custid')['gender']
tr_tx['g']=tr_tx['custid'].map(ymap); tr_tx['fold']=tr_tx['custid'].map(fold)
gmean=ymap.mean()

CAT=['brd_nm','corner_nm','pc_nm','part_nm','str_nm','buyer_nm']
NUM=['tot_amt','dis_amt','net_amt','inst_mon','inst_fee','hour','weekday','month','is_weekend']

# 빈도 인코딩 (비지도)
def freq_encode(col):
    vc=pd.concat([tr_tx[col],te_tx[col]]).astype(str).value_counts()
    return tr_tx[col].astype(str).map(vc).fillna(0).values, te_tx[col].astype(str).map(vc).fillna(0).values

# 거래레벨 TE (고객 fold 기준 OOF — 누수차단)
def tx_te(col, sm=30):
    oof=np.full(len(tr_tx),np.nan); test=np.zeros(len(te_tx))
    for i in range(N_SPLITS):
        sub=tr_tx[tr_tx['fold']!=i]
        s=sub.groupby(col)['g'].agg(['mean','count']); enc=(s['mean']*s['count']+gmean*sm)/(s['count']+sm)
        m=tr_tx['fold']==i
        oof[m.values]=tr_tx.loc[m,col].map(enc).fillna(gmean).values
    s=tr_tx.groupby(col)['g'].agg(['mean','count']); enc=(s['mean']*s['count']+gmean*sm)/(s['count']+sm)
    test=te_tx[col].map(enc).fillna(gmean).values
    return oof, test

# 거래레벨 피쳐 행렬 구성
Xtr=pd.DataFrame(index=tr_tx.index); Xte=pd.DataFrame(index=te_tx.index)
for c in NUM: Xtr[c]=tr_tx[c].fillna(0).values; Xte[c]=te_tx[c].fillna(0).values
for c in CAT:
    a,b=freq_encode(c); Xtr[f'freq_{c}']=a; Xte[f'freq_{c}']=b
    a,b=tx_te(c);       Xtr[f'te_{c}']=a;   Xte[f'te_{c}']=b      # ← TE 추가 (강화)
Xtr['disc_rate']=tr_tx['dis_amt']/(tr_tx['tot_amt'].abs()+1); Xte['disc_rate']=te_tx['dis_amt']/(te_tx['tot_amt'].abs()+1)
Xtr['amt_per_inst']=tr_tx['net_amt']/(tr_tx['inst_mon'].clip(lower=1)); Xte['amt_per_inst']=te_tx['net_amt']/(te_tx['inst_mon'].clip(lower=1))
print('거래레벨 피쳐:', Xtr.shape[1])

# 거래레벨 OOF (강화: 트리↑ lr↓)
fvals=tr_tx['fold'].values; tx_oof=np.zeros(len(tr_tx)); tx_test=np.zeros(len(te_tx))
for i in range(N_SPLITS):
    trm=fvals!=i; vam=fvals==i
    m=lgb.LGBMClassifier(n_estimators=700, learning_rate=0.03, num_leaves=127,
        subsample=0.8, colsample_bytree=0.7, reg_lambda=3.0, min_child_samples=200,
        n_jobs=-1, max_bin=255, verbose=-1)
    m.fit(Xtr.values[trm], tr_tx['g'].values[trm])
    tx_oof[vam]=m.predict_proba(Xtr.values[vam])[:,1]
    tx_test+=m.predict_proba(Xte.values)[:,1]/N_SPLITS
print('거래레벨 OOF AUC (강화):', round(roc_auc_score(tr_tx['g'], tx_oof),5), '(이전 0.62747)')

# 집계 — 전체 + 카테고리별(남/여 카테고리 거래의 확률 분리)
cat=(tr_tx['part_nm'].astype(str)+tr_tx['corner_nm'].astype(str))
catE=(te_tx['part_nm'].astype(str)+te_tx['corner_nm'].astype(str))
tr_tx['_mc']=cat.str.contains('남성|신사|골프|스포츠').astype(int)
te_tx['_mc']=catE.str.contains('남성|신사|골프|스포츠').astype(int)

def agg_all(txdf, probs):
    d=pd.DataFrame({'custid':txdf['custid'].values,'p':probs,'mc':txdf['_mc'].values})
    d['male']=(d['p']>0.5).astype(int)
    g=d.groupby('custid')['p']
    out=g.agg(['mean','std','max','min','median']); out.columns=['txp_'+c for c in out.columns]
    out['txp_q25']=g.quantile(.25); out['txp_q75']=g.quantile(.75); out['txp_spread']=out['txp_q75']-out['txp_q25']
    out['txp_frac_male']=d.groupby('custid')['male'].mean()
    # 카테고리별 평균확률 (남성카테고리 거래 vs 나머지)
    out['txp_mc_mean']=d[d.mc==1].groupby('custid')['p'].mean()
    out['txp_nonmc_mean']=d[d.mc==0].groupby('custid')['p'].mean()
    out['txp_mc_gap']=out['txp_mc_mean']-out['txp_nonmc_mean']
    return out.fillna(gmean)

stk_tr=agg_all(tr_tx, tx_oof); stk_te=agg_all(te_tx, tx_test)
stack=pd.concat([stk_tr,stk_te])
print('stacking 피쳐:', stack.shape[1])

tr_ids=y_train['custid'].values
def oof_of(feats):
    X=feats.loc[tr_ids]; t=ymap.loc[tr_ids].values; fv=pd.Series(tr_ids,index=tr_ids).map(fold).values
    Xv=X.values.astype(np.float32); o=np.zeros(len(Xv))
    for i in range(N_SPLITS):
        m=lgb.LGBMClassifier(n_estimators=400,learning_rate=0.03,num_leaves=31,subsample=0.8,colsample_bytree=0.6,reg_lambda=2.0,n_jobs=-1,max_bin=127,verbose=-1)
        m.fit(Xv[fv!=i],t[fv!=i]); o[fv==i]=m.predict_proba(Xv[fv==i])[:,1]
    return roc_auc_score(t,o)
print('BASE 439              :', round(oof_of(allf),5))
print('439 + tx-stacking강화 :', round(oof_of(allf.join(stack)),5), '(이전 0.71459)')
stack.to_parquet(f'{FEAT_DIR}/tx_stack_v2.parquet')

거래레벨 피쳐: 23
거래레벨 OOF AUC (강화): 0.62889 (이전 0.62747)
stacking 피쳐: 12
BASE 439              : 0.71373
439 + tx-stacking강화 : 0.71625 (이전 0.71459)


In [5]:
# === tx-stacking v3: 집계 축 확장 (이전 셀 변수 재사용) ===

# 거래에 여러 축 플래그 부착
def add_axes(tx):
    cat=(tx['part_nm'].astype(str)+tx['corner_nm'].astype(str)+tx['pc_nm'].astype(str))
    amt=tx['tot_amt'].clip(lower=0)
    tx['ax_male']=cat.str.contains('남성|신사|골프|스포츠').astype(int)
    tx['ax_fem']=cat.str.contains('여성|숙녀|화장품|란제리|미씨|영캐').astype(int)
    tx['ax_kid']=cat.str.contains('아동|유아|완구').astype(int)
    tx['ax_hi']=(amt>300000).astype(int)
    tx['ax_lo']=(amt<50000).astype(int)
    tx['ax_eve']=(tx['hour']>=18).astype(int)
    tx['ax_wend']=tx['is_weekend']
    tx['amt']=amt
    return tx
tr_tx=add_axes(tr_tx); te_tx=add_axes(te_tx)
AXES=['ax_male','ax_fem','ax_kid','ax_hi','ax_lo','ax_eve','ax_wend']

def agg_v3(txdf, probs):
    d=pd.DataFrame({'custid':txdf['custid'].values,'p':probs,'amt':txdf['amt'].values})
    for a in AXES: d[a]=txdf[a].values
    g=d.groupby('custid'); out=pd.DataFrame(index=g.size().index)
    # 전체 분포
    gp=g['p']; out['txp_mean']=gp.mean(); out['txp_std']=gp.std(); out['txp_max']=gp.max()
    out['txp_min']=gp.min(); out['txp_median']=gp.median()
    out['txp_q25']=gp.quantile(.25); out['txp_q75']=gp.quantile(.75); out['txp_spread']=out['txp_q75']-out['txp_q25']
    out['txp_frac_male']=d.assign(m=(d.p>0.5).astype(int)).groupby('custid')['m'].mean()
    # 금액 가중 평균확률 (본인용 추정 = 비싼거 가중)
    out['txp_amtw']=(g.apply(lambda x:(x['p']*x['amt']).sum()/(x['amt'].sum()+1)) if False else
                     d.assign(pa=d.p*d.amt).groupby('custid')['pa'].sum()/(g['amt'].sum()+1))
    # 축별 평균확률 + 전체평균과의 gap
    base_mean=out['txp_mean']
    for a in AXES:
        sub=d[d[a]==1].groupby('custid')['p'].mean()
        out[f'txp_{a}_mean']=sub.reindex(out.index)
        out[f'txp_{a}_gap']=(sub.reindex(out.index)-base_mean)
    return out.fillna(gmean)

stk_tr=agg_v3(tr_tx, tx_oof); stk_te=agg_v3(te_tx, tx_test)
stack_v3=pd.concat([stk_tr,stk_te]).fillna(gmean)
print('v3 stacking 피쳐:', stack_v3.shape[1])
print('BASE 439              :', round(oof_of(allf),5))
print('439 + tx-stacking v3  :', round(oof_of(allf.join(stack_v3)),5), '(v2 0.71625)')
stack_v3.to_parquet(f'{FEAT_DIR}/tx_stack_v3.parquet')

v3 stacking 피쳐: 24
BASE 439              : 0.71373
439 + tx-stacking v3  : 0.71739 (v2 0.71625)


In [8]:

import pandas as pd, numpy as np
from autogluon.tabular import TabularPredictor

allf = pd.read_parquet(f'{FEAT_DIR}/feats_1st_plus_8adv.parquet')
stack = pd.read_parquet(f'{FEAT_DIR}/tx_stack_v3.parquet')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv')

allf2 = allf.join(stack).fillna(0)          # 439 + 24 = 463
allf2.columns = [str(c) for c in allf2.columns]
print('피쳐수:', allf2.shape[1])

tr_ids = y_train['custid'].values
test_ids = [c for c in allf2.index if c not in set(tr_ids)]

train_data = allf2.loc[tr_ids].copy()
train_data['gender'] = y_train.set_index('custid')['gender'].values
X_test = allf2.loc[test_ids]

predictor = TabularPredictor(
    label='gender', problem_type='binary', eval_metric='roc_auc'
).fit(train_data, presets='best_quality', time_limit=3600)   # 1시간 고정

pred = predictor.predict_proba(X_test)[1]
pd.DataFrame({'custid': X_test.index, 'gender': pred}).to_csv(
    f'{FEAT_DIR}/submission_463_txstack.csv', index=False)
print('OOF:', predictor.leaderboard(silent=True).iloc[0]['score_val'])
print('saved submission_463_txstack.csv')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 7.4 MB/s eta 0:00:00
피쳐수: 463


No path specified. Models will be saved in: "AutogluonModels/ag-20260606_083827"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       8.64 GB / 12.67 GB (68.2%)
Disk Space Avail:   87.02 GB / 107.72 GB (80.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stack

OOF: 0.7231100647242943
saved submission_463_txstack.csv


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import xgboost as xgb

# === v4: 2-way 축교차 + XGB 거래모델 추가 ===
# (이전 셀의 tr_tx, te_tx, Xtr, Xte, tx_oof, tx_test, AXES, allf, oof_of, fold, ymap, gmean 재사용)

# --- (a) XGB로 거래레벨 OOF 하나 더 (다른 시각) ---
fvals = tr_tx['fold'].values
xgb_oof = np.zeros(len(tr_tx)); xgb_test = np.zeros(len(te_tx))
for i in range(5):
    trm = fvals!=i; vam = fvals==i
    m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=500,
        learning_rate=0.04, max_depth=7, subsample=0.8, colsample_bytree=0.6,
        reg_lambda=3.0, eval_metric='auc', verbosity=0)
    m.fit(Xtr.values[trm], tr_tx['g'].values[trm])
    xgb_oof[vam] = m.predict_proba(Xtr.values[vam])[:,1]
    xgb_test += m.predict_proba(Xte.values)[:,1]/5
from sklearn.metrics import roc_auc_score
print('XGB 거래레벨 OOF:', round(roc_auc_score(tr_tx['g'], xgb_oof),5))

# --- (b) 2-way 축교차 플래그 ---
def add_cross(tx):
    tx['cx_male_hi'] = ((tx['ax_male']==1)&(tx['ax_hi']==1)).astype(int)   # 남성 비싼거=본인 강신호
    tx['cx_fem_hi']  = ((tx['ax_fem']==1)&(tx['ax_hi']==1)).astype(int)
    tx['cx_male_eve']= ((tx['ax_male']==1)&(tx['ax_eve']==1)).astype(int)
    tx['cx_fem_lo']  = ((tx['ax_fem']==1)&(tx['ax_lo']==1)).astype(int)
    return tx
tr_tx = add_cross(tr_tx); te_tx = add_cross(te_tx)
CROSS = ['cx_male_hi','cx_fem_hi','cx_male_eve','cx_fem_lo']

# --- (c) 집계: LGB확률 + XGB확률 둘 다, 2-way 축 포함 ---
def agg_v4(txdf, p_lgb, p_xgb):
    d = pd.DataFrame({'custid':txdf['custid'].values,'p':p_lgb,'px':p_xgb,'amt':txdf['amt'].values})
    for a in (AXES+CROSS): d[a]=txdf[a].values
    g = d.groupby('custid'); out = pd.DataFrame(index=g.size().index)
    # XGB 확률 분포 (새 시각)
    gx=g['px']; out['txx_mean']=gx.mean(); out['txx_std']=gx.std(); out['txx_max']=gx.max(); out['txx_min']=gx.min()
    out['txx_frac_male']=d.assign(m=(d.px>0.5).astype(int)).groupby('custid')['m'].mean()
    # LGB-XGB 합의도 (두 모델 일치할수록 확신)
    out['tx_agree']=(d.assign(ag=np.abs(d.p-d.px)).groupby('custid')['ag'].mean())
    base_mean=g['p'].mean()
    # 2-way 축별 평균확률 + gap
    for a in CROSS:
        sub=d[d[a]==1].groupby('custid')['p'].mean()
        out[f'txp_{a}_mean']=sub.reindex(out.index); out[f'txp_{a}_gap']=sub.reindex(out.index)-base_mean
    return out.fillna(gmean)

stk_tr = agg_v4(tr_tx, tx_oof, xgb_oof); stk_te = agg_v4(te_tx, tx_test, xgb_test)
stack_v4_extra = pd.concat([stk_tr, stk_te]).fillna(gmean)

# v3 + v4추가분 합본
stack_v3 = pd.read_parquet(f"{FEAT_DIR}/tx_stack_v3.parquet")
stack_v4 = stack_v3.join(stack_v4_extra).fillna(gmean)
print('v4 stacking 피쳐:', stack_v4.shape[1], '(v3 24 + 추가', stack_v4_extra.shape[1],')')

tr_ids=ymap.index.values if False else pd.read_csv(f'{DATA_DIR}/y_train.csv')['custid'].values
print('BASE 439          :', round(oof_of(allf),5))
print('439 + v3          :', round(oof_of(allf.join(stack_v3)),5))
print('439 + v4          :', round(oof_of(allf.join(stack_v4)),5), '(v3 0.71739)')
stack_v4.to_parquet(f'{FEAT_DIR}/tx_stack_v4.parquet')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
XGB 거래레벨 OOF: 0.62774
v4 stacking 피쳐: 38 (v3 24 + 추가 14 )
BASE 439          : 0.71373
439 + v3          : 0.71739
439 + v4          : 0.71766 (v3 0.71739)
